###  This section is a EDA for the `plant` with `most remissions (512)` in order to make an `early preparation` and `testing` (temporal feature engineering first) for the final model. 

##### The model will be TFT (Temporal Fusion Transformer) which is a specialized deep neural network that uses self-attention to capture complex dependencies between features and time steps.

###### Check `model_justification.md` for more details about the model selection.

In [2]:
# Uncomment if needed:
# %pip install torch pytorch-forecasting lightning

import pandas as pd
import torch
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor

DATA_PATH = "../data/processed/remissions_db_cleaned.csv"
PLANT_CODE = "512"

df = pd.read_csv(DATA_PATH)

required_cols = {"ship_plant_code", "u_Volumen"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["ship_plant_code"] = df["ship_plant_code"].astype(str)
df = df[df["ship_plant_code"] == PLANT_CODE].copy()

time_col = "start_time" if "start_time" in df.columns else "order_date"
if time_col not in df.columns:
    raise ValueError("No time column found. Expected 'start_time' or 'order_date'.")

df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
df["u_Volumen"] = pd.to_numeric(df["u_Volumen"], errors="coerce")
df = df.dropna(subset=[time_col, "u_Volumen"])

freq = "h" if time_col == "start_time" else "D"
df = (
    df.set_index(time_col)
      .resample(freq)["u_Volumen"]
      .sum()
      .reset_index()
      .rename(columns={time_col: "time"})
)
df["ship_plant_code"] = PLANT_CODE

df["time_idx"] = (
    (df["time"] - df["time"].min()) / pd.Timedelta(1, unit="h" if freq == "h" else "D")
).astype(int)
df["dayofweek"] = df["time"].dt.dayofweek.astype("int16")
df["month"] = df["time"].dt.month.astype("int16")
df["day"] = df["time"].dt.day.astype("int16")
if freq == "H":
    df["hour"] = df["time"].dt.hour.astype("int16")

print(f"Rows: {len(df)} | Range: {df['time'].min()} -> {df['time'].max()} | Freq: {freq}")
df.head()

Rows: 54927 | Range: 2020-02-04 05:00:00 -> 2026-05-11 19:00:00 | Freq: h


,time,u_Volumen,ship_plant_code,time_idx,dayofweek,month,day
0,2020-02-04 05:00:00,17.0,512,0,1,2,4
1,2020-02-04 06:00:00,0.0,512,1,1,2,4
2,2020-02-04 07:00:00,0.0,512,2,1,2,4
3,2020-02-04 08:00:00,0.0,512,3,1,2,4
4,2020-02-04 09:00:00,4.5,512,4,1,2,4


In [ ]:
if freq == "h":
    max_prediction_length = 24
    max_encoder_length = 7 * 24
    known_reals = ["time_idx", "hour", "dayofweek", "month", "day"]
else:
    max_prediction_length = 7
    max_encoder_length = 30
    known_reals = ["time_idx", "dayofweek", "month", "day"]

min_required = max_encoder_length + max_prediction_length + 1
if len(df) < min_required:
    raise ValueError(f"Not enough rows for TFT. Need at least {min_required}, got {len(df)}.")

training_cutoff = df["time_idx"].max() - max_prediction_length

if freq == "h" and "hour" not in df.columns:
    df["hour"] = df["time"].dt.hour.astype("int16")

known_reals_filtered = [col for col in known_reals if col in df.columns]

training = TimeSeriesDataSet(
    df[df["time_idx"] <= training_cutoff],
    time_idx="time_idx",
    target="u_Volumen",
    group_ids=["ship_plant_code"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["ship_plant_code"],
    time_varying_known_reals=known_reals_filtered,
    time_varying_unknown_reals=["u_Volumen"],
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(),
    optimizer="adam",
    reduce_on_plateau_patience=4,
)

early_stop = EarlyStopping(monitor="val_loss", patience=3, mode="min")
lr_monitor = LearningRateMonitor()

trainer = Trainer(
    max_epochs=10,
    accelerator="auto",
    gradient_clip_val=0.1,
    callbacks=[early_stop, lr_monitor],
    enable_checkpointing=False,
)

trainer.fit(tft, train_dataloader, val_dataloader)

predictions = tft.predict(val_dataloader)
actuals = torch.cat([y[0] for x, y in iter(val_dataloader)])
mae = (predictions - actuals).abs().mean()
print(f"MAE: {mae.item():.4f}")

raw_predictions, x = tft.predict(val_dataloader, mode="raw", return_x=True)
tft.plot_prediction(x, raw_predictions, idx=0)

C:\Users\gilin\AppData\Roaming\Python\Python313\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\Users\gilin\AppData\Roaming\Python\Python313\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      1 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    160 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  1.8 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  4.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  3.7 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    119 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 23.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 396                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

C:\Users\gilin\AppData\Roaming\Python\Python313\site-packages\lightning\pytorch\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

...

C:\Users\gilin\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does
not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(

C:\Users\gilin\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does
not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(

C:\Users\gilin\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does
not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(

C:\Users\gilin\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does
not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(

C:\Users\gilin\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does
not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(